# Step 13: Per-Employee Skill Gap Engine

## Overview
This notebook executes the per-employee skill gap engine via set subtraction:
$$\text{Missing Skills} = \text{Required Role Skills} \setminus \text{Possessed Employee Skills}$$

Each missing skill is weighted by its O*NET importance rating (`Data Value`), producing a `Weighted Skill Gap Score` per employee.


In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join("..", "data", "processed")

emp_skills = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_current_skills.csv"))
role_master = pd.read_csv(os.path.join(PROCESSED_DIR, "role_intelligence_master.csv"))
ess_df = pd.read_csv(os.path.join(PROCESSED_DIR, "essential_skills_processed.csv"))
soft_df = pd.read_csv(os.path.join(PROCESSED_DIR, "software_skills_processed.csv"))
attr_df = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_attrition_processed.csv"))

print("Loaded all inputs successfully.")


Loaded all inputs successfully.


---
## 1. Execute Set Subtraction & Weighted Skill Gap Calculation


In [2]:
# Pre-build importance lookup dictionary for essential skills
importance_lookup = ess_df.set_index(['O*NET-SOC Code', 'Element Name'])['Data Value'].to_dict()

# Create role-level required skills maps
required_skills_by_soc = {}
for soc in role_master['ONET_SOC_Code'].unique():
    ess_req = ess_df[ess_df['O*NET-SOC Code'] == soc]['Element Name'].unique()
    soft_req = soft_df[soft_df['O*NET-SOC Code'] == soc]['Normalized_Tool_Name'].unique()
    required_skills_by_soc[soc] = set(ess_req).union(set(soft_req))

# Group possessed skills by employee
possessed_skills_by_emp = emp_skills.groupby('EmployeeNumber')['Skill_Name'].apply(set).to_dict()

gap_records = []

# Merge employee role mapping
emp_soc_map = pd.merge(attr_df[['EmployeeNumber', 'JobRole']], role_master[['HR_Job_Role', 'ONET_SOC_Code']], left_on='JobRole', right_on='HR_Job_Role', how='left').set_index('EmployeeNumber')['ONET_SOC_Code'].to_dict()

for emp_id, soc_code in emp_soc_map.items():
    required = required_skills_by_soc.get(soc_code, set())
    possessed = possessed_skills_by_emp.get(emp_id, set())
    
    missing = required - possessed
    
    # Calculate weighted gap score
    weighted_score = 0.0
    for sk in missing:
        weight = importance_lookup.get((soc_code, sk), 3.0) # Default weight 3.0 for software tools
        weighted_score += weight
        
        gap_records.append({
            'EmployeeNumber': emp_id,
            'ONET_SOC_Code': soc_code,
            'Missing_Skill_Name': sk,
            'Importance_Weight': weight
        })

gaps_detail_df = pd.DataFrame(gap_records)
print(f"Total Individual Skill Gap Records Identified: {len(gaps_detail_df)}")

# Summary per employee
emp_gap_summary = gaps_detail_df.groupby('EmployeeNumber').agg(
    Missing_Skills_Count=('Missing_Skill_Name', 'count'),
    Total_Weighted_Gap_Score=('Importance_Weight', 'sum')
).reset_index()

print("\nSample Per-Employee Skill Gap Summary:")
print(emp_gap_summary.head(10).to_string(index=False))

out_path_detail = os.path.join(PROCESSED_DIR, "employee_skill_gaps_detail.csv")
out_path_summary = os.path.join(PROCESSED_DIR, "employee_skill_gaps_summary.csv")

gaps_detail_df.to_csv(out_path_detail, index=False)
emp_gap_summary.to_csv(out_path_summary, index=False)

print(f"\nSaved Details: {out_path_detail}")
print(f"Saved Summary: {out_path_summary}")


Total Individual Skill Gap Records Identified: 56271

Sample Per-Employee Skill Gap Summary:
 EmployeeNumber  Missing_Skills_Count  Total_Weighted_Gap_Score
              1                    52                    156.75
              2                    36                    110.38
              4                    14                     42.50
              5                    36                    110.88
              7                    14                     43.00
              8                    14                     42.25
             10                    14                     41.75
             11                    14                     42.87
             12                    53                    161.50
             13                    26                     77.13

Saved Details: ..\data\processed\employee_skill_gaps_detail.csv
Saved Summary: ..\data\processed\employee_skill_gaps_summary.csv
